# P06 Human Audit 30 Reproducible

由原 P06 拆出：30 筆人工查核的抽樣、網頁填表與結果圖表。


# P06 補充：30 筆人工查核：分層抽樣、網頁填表與論文圖表

目的：
1. 將原先 15 筆人工稽核擴充為 30 筆。
2. 抽樣時同時涵蓋不同溫度、場合、活動量、風格，以及高分、低分與模型衝突案例。
3. 人工查核仍沿用既有 C* checklist，逐項判斷 YES / NO，不新增內容重複的稽核清單。
4. 修正舊圖表的 XY 軸，使座標軸明確標示變數名稱、尺度與單位。

使用方式：
1. 先執行到「產生 30 筆抽樣與人工查核網頁」。
2. 用瀏覽器打開 `human_audit_30_form.html`，逐筆填 YES / NO，匯出 CSV。
3. 將匯出的 CSV 放到 `A38_human_audit_30_manual_results.csv`。
4. 再執行「讀取人工結果並產生分析圖表」。

編號說明：A30-A34 已由第二模型補充實驗使用，因此本段 30 筆人工查核從 A35 接續；人工填寫模板 A38 放在 human_audit_30 資料夾，填完後另存為正式人工結果 CSV，再重跑本 notebook 產生後續分析圖表。


In [4]:
from __future__ import annotations

import csv
import html
import json
import math
import random
import re
from collections import Counter, defaultdict
from pathlib import Path
from statistics import mean, stdev
from typing import Dict, Iterable, List, Optional, Tuple

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np


# ------------------------------------------------------------
# 1. 路徑與基本設定
# ------------------------------------------------------------
ROOT = Path("/home/ester/valid_description")
POLYVORE_DIR = ROOT / "polyvore_data" / "polyvore_outfits"
DISJOINT_DIR = POLYVORE_DIR / "disjoint"
IMAGE_DIR = POLYVORE_DIR / "images"

JOURNAL_DIR = ROOT / "experiment data" / "journal_figures_tables_20260629"
SOURCE_DIR = JOURNAL_DIR / "source_programs"
DERIVED_DIR = JOURNAL_DIR / "derived_audits"
TABLE_DIR = JOURNAL_DIR / "tables"
FIG_DIR = JOURNAL_DIR / "figures"
HUMAN_DIR = JOURNAL_DIR / "human_audit_30"

for directory in [DERIVED_DIR, TABLE_DIR, FIG_DIR, HUMAN_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_TOTAL = 30
N_PER_STRATUM = 10
MAX_RANK_GAP_FOR_CONSENSUS = 0.35
MIN_RANK_GAP_FOR_CONFLICT = 0.50
MIN_SCORE_GAP_FOR_CONFLICT = 0.25

QWEN_SCORE_PATH = ROOT / "Qwen3_Instruct" / "phase3_scores_Qwen3VL32B.jsonl"
GEMMA_SCORE_PATH = ROOT / "Gemma3" / "phase3_scores_Gemma3.jsonl"
QWEN_CSTAR_PATH = ROOT / "Qwen3_Instruct" / "checklist_C_star_Qwen3VL32B.json"
GEMMA_CSTAR_PATH = ROOT / "Gemma3" / "checklist_C_star_Gemma3.json"

ORIGINAL_TITLE_PATH = POLYVORE_DIR / "polyvore_outfit_titles.json"
GENERATED_TITLE_PATH = ROOT / "new_polyvore_outfit_titles.json"
GENERATED_ABLATION_PATH = ROOT / "new_polyvore_outfit_titles_with_ablation.json"

SELECTED_CASES_CSV = DERIVED_DIR / "A36_human_audit_30_selected_cases.csv"
SELECTED_CASES_JSON = DERIVED_DIR / "A37_human_audit_30_selected_cases.json"
MANUAL_TEMPLATE_CSV = HUMAN_DIR / "A38_human_audit_30_manual_results_template.csv"
MANUAL_RESULTS_CSV = HUMAN_DIR / "A38_human_audit_30_manual_results.csv"
FORM_HTML = HUMAN_DIR / "human_audit_30_form.html"

print("人工查核輸出資料夾：", HUMAN_DIR)
print("人工結果 CSV 預期路徑：", MANUAL_RESULTS_CSV)


人工查核輸出資料夾： /home/ester/valid_description/experiment data/journal_figures_tables_20260629/human_audit_30
人工結果 CSV 預期路徑： /home/ester/valid_description/experiment data/journal_figures_tables_20260629/human_audit_30/A38_human_audit_30_manual_results.csv


In [5]:
# ------------------------------------------------------------
# 2. 通用讀寫工具
# ------------------------------------------------------------
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_jsonl_scores(path: Path) -> Dict[str, dict]:
    rows = {}
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if obj.get("score") is None:
                continue
            set_id = str(obj["id"])
            decision_map = {
                str(item.get("id")): str(item.get("answer", "")).strip().lower()
                for item in obj.get("decisions", [])
            }
            rows[set_id] = {
                "set_id": set_id,
                "score": float(obj["score"]),
                "text": obj.get("text", ""),
                "feedback": obj.get("feedback", ""),
                "rewrite": obj.get("rewrite", ""),
                "decisions": decision_map,
            }
    return rows


def write_csv(path: Path, rows: List[dict], fieldnames: Optional[List[str]] = None):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not fieldnames:
        keys = []
        seen = set()
        for row in rows:
            for key in row.keys():
                if key not in seen:
                    seen.add(key)
                    keys.append(key)
        fieldnames = keys
    with path.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)
    print("[寫入]", path)


def read_csv_rows(path: Path) -> List[dict]:
    with path.open("r", newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


def pct_rank(values_by_id: Dict[str, float]) -> Dict[str, float]:
    """回傳百分位排名。分數越低，rank 越接近 0；分數越高，rank 越接近 1。"""
    items = sorted(values_by_id.items(), key=lambda kv: kv[1])
    n = len(items)
    if n <= 1:
        return {items[0][0]: 1.0} if items else {}

    ranks = {}
    i = 0
    while i < n:
        j = i
        while j + 1 < n and items[j + 1][1] == items[i][1]:
            j += 1
        avg_rank_1_based = (i + 1 + j + 1) / 2
        pct = (avg_rank_1_based - 1) / (n - 1)
        for k in range(i, j + 1):
            ranks[items[k][0]] = pct
        i = j + 1
    return ranks


def safe_float(value, default=None):
    try:
        if value is None or value == "":
            return default
        return float(value)
    except Exception:
        return default


In [6]:
# ------------------------------------------------------------
# 3. 載入資料、C* checklist 與環境因子
# ------------------------------------------------------------
qwen_scores = load_jsonl_scores(QWEN_SCORE_PATH)
gemma_scores = load_jsonl_scores(GEMMA_SCORE_PATH)
qwen_checklist = load_json(QWEN_CSTAR_PATH)["checklist"]
gemma_checklist = load_json(GEMMA_CSTAR_PATH)["checklist"]
original_titles = load_json(ORIGINAL_TITLE_PATH)
generated_titles = load_json(GENERATED_TITLE_PATH)
generated_ablation = load_json(GENERATED_ABLATION_PATH)
metadata = load_json(POLYVORE_DIR / "polyvore_item_metadata.json")

split_outfits = {}
for split in ["train", "valid", "test"]:
    for outfit in load_json(DISJOINT_DIR / f"{split}.json"):
        split_outfits[str(outfit["set_id"])] = outfit

thermal_rows = {}
for split in ["train", "valid", "test"]:
    path = ROOT / "熱舒適模型" / f"{split}_temperature.json"
    for row in load_json(path):
        thermal_rows[str(row["set_id"])] = row

common_ids = sorted(set(qwen_scores) & set(gemma_scores) & set(generated_titles))

print(f"Qwen scores：{len(qwen_scores):,}")
print(f"Gemma scores：{len(gemma_scores):,}")
print(f"共同可抽樣 set_id：{len(common_ids):,}")
print(f"Qwen C* items：{len(qwen_checklist)}")
print(f"Gemma C* items：{len(gemma_checklist)}")


Qwen scores：35,140
Gemma scores：35,140
共同可抽樣 set_id：35,140
Qwen C* items：10
Gemma C* items：15


In [7]:
# ------------------------------------------------------------
# 4. 文字解析：溫度、場合、活動量、風格
# ------------------------------------------------------------
TEMP_PATTERN = re.compile(r"(-?\d+(?:\.\d+)?)\s*°?\s*C", flags=re.IGNORECASE)


def parse_temperature_from_text(text: str) -> Optional[float]:
    m = TEMP_PATTERN.search(text or "")
    return float(m.group(1)) if m else None


def temperature_bin(temp_c: Optional[float]) -> str:
    if temp_c is None:
        return "unknown"
    if temp_c < 12:
        return "cold (<12 degC)"
    if temp_c < 18:
        return "cool (12-18 degC)"
    if temp_c < 24:
        return "mild (18-24 degC)"
    return "warm/hot (>=24 degC)"


def met_bin(met: Optional[float]) -> str:
    if met is None:
        return "unknown"
    if met < 1.3:
        return "low activity (<1.3 MET)"
    if met < 1.6:
        return "moderate activity (1.3-1.6 MET)"
    return "higher activity (>=1.6 MET)"


OCCASION_KEYWORDS = {
    "formal/work": ["office", "work", "meeting", "business", "formal", "conference", "interview", "professional"],
    "evening/event": ["evening", "night", "dinner", "date", "party", "wedding", "cocktail", "concert", "gala", "romantic"],
    "casual/daily": ["casual", "daily", "day out", "coffee", "brunch", "weekend", "errand", "school", "relaxed"],
    "outdoor/travel": ["beach", "vacation", "travel", "picnic", "park", "outdoor", "festival", "walk", "walking", "spring day"],
}

STYLE_KEYWORDS = {
    "elegant/romantic": ["elegant", "romantic", "glam", "glamorous", "chic", "sophisticated", "velvet", "lace"],
    "casual/street": ["casual", "street", "cool-girl", "denim", "sneaker", "effortless"],
    "boho/beach": ["boho", "bohemian", "beach", "island", "breezy", "summer"],
    "sporty/active": ["sporty", "active", "athletic", "hike", "running", "workout"],
    "dark/edgy": ["goth", "dark", "edgy", "black", "leather", "rock"],
    "minimal/classic": ["minimal", "classic", "preppy", "clean", "simple", "tailored"],
}


def infer_label(text: str, keyword_map: Dict[str, List[str]], default: str = "other/unclear") -> str:
    lower = (text or "").lower()
    scores = []
    for label, keywords in keyword_map.items():
        count = sum(1 for keyword in keywords if keyword in lower)
        if count:
            scores.append((count, label))
    if not scores:
        return default
    scores.sort(reverse=True)
    return scores[0][1]


def item_summary_for_set(set_id: str) -> List[dict]:
    outfit = split_outfits.get(str(set_id), {})
    items = []
    for item in outfit.get("items", []):
        item_id = str(item["item_id"])
        meta = metadata.get(item_id, {})
        image_rel = f"../../../polyvore_data/polyvore_outfits/images/{item_id}.jpg"
        items.append({
            "item_id": item_id,
            "index": item.get("index"),
            "semantic_category": meta.get("semantic_category", ""),
            "category_id": meta.get("category_id", ""),
            "title": meta.get("title") or meta.get("url_name") or "",
            "image_src": image_rel,
        })
    return items


def original_text_for_set(set_id: str) -> str:
    obj = original_titles.get(str(set_id), {})
    title = obj.get("title", "")
    url_name = obj.get("url_name", "")
    return " ".join(part for part in [url_name, title] if part).strip()


In [8]:
# ------------------------------------------------------------
# 5. 建立抽樣母體
# ------------------------------------------------------------
rank_qwen = pct_rank({sid: qwen_scores[sid]["score"] for sid in common_ids})
rank_gemma = pct_rank({sid: gemma_scores[sid]["score"] for sid in common_ids})

sampling_frame = []
for set_id in common_ids:
    q = qwen_scores[set_id]
    g = gemma_scores[set_id]
    title = generated_titles.get(set_id, {}).get("title") or q.get("text", "")
    thermal = thermal_rows.get(set_id, {})
    temp = safe_float(thermal.get("TSUB_target_C"))
    if temp is None:
        temp = parse_temperature_from_text(title)
    met = safe_float(thermal.get("met_value"))
    activity = thermal.get("activity", "")
    rq = rank_qwen[set_id]
    rg = rank_gemma[set_id]
    combined_rank = (rq + rg) / 2
    row = {
        "set_id": set_id,
        "score_qwen": q["score"],
        "score_gemma": g["score"],
        "rank_qwen": rq,
        "rank_gemma": rg,
        "combined_rank": combined_rank,
        "rank_gap": abs(rq - rg),
        "score_gap": abs(q["score"] - g["score"]),
        "temperature_c": temp,
        "temp_bin": temperature_bin(temp),
        "met_value": met,
        "met_bin": met_bin(met),
        "activity": activity,
        "occasion_bin": infer_label(title, OCCASION_KEYWORDS),
        "style_bin": infer_label(title, STYLE_KEYWORDS),
        "generated_title": title,
        "generated_no_weather": generated_ablation.get(set_id, {}).get("no_weather", ""),
        "generated_no_occasion": generated_ablation.get(set_id, {}).get("no_occasion", ""),
        "generated_no_style": generated_ablation.get(set_id, {}).get("no_style", ""),
        "original_text": original_text_for_set(set_id),
    }
    sampling_frame.append(row)

write_csv(DERIVED_DIR / "A35_human_audit_30_sampling_frame.csv", sampling_frame)

print("抽樣母體：", len(sampling_frame))
print("溫度分布：", Counter(row["temp_bin"] for row in sampling_frame))
print("場合分布：", Counter(row["occasion_bin"] for row in sampling_frame).most_common())
print("活動量分布：", Counter(row["met_bin"] for row in sampling_frame))
print("風格分布：", Counter(row["style_bin"] for row in sampling_frame).most_common())


[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A35_human_audit_30_sampling_frame.csv
抽樣母體： 35140
溫度分布： Counter({'mild (18-24 degC)': 20119, 'warm/hot (>=24 degC)': 10071, 'cool (12-18 degC)': 3830, 'cold (<12 degC)': 1120})
場合分布： [('other/unclear', 16145), ('evening/event', 8979), ('casual/daily', 4704), ('outdoor/travel', 4048), ('formal/work', 1264)]
活動量分布： Counter({'low activity (<1.3 MET)': 25786, 'moderate activity (1.3-1.6 MET)': 7870, 'higher activity (>=1.6 MET)': 1484})
風格分布： [('elegant/romantic', 12879), ('casual/street', 10570), ('boho/beach', 4889), ('other/unclear', 3846), ('dark/edgy', 1752), ('sporty/active', 796), ('minimal/classic', 408)]


In [9]:
# ------------------------------------------------------------
# 6. 30 筆分層抽樣：高分、低分、衝突案例
# ------------------------------------------------------------
random.seed(RANDOM_SEED)

low_pool = [
    row for row in sampling_frame
    if row["combined_rank"] <= 0.25 and row["rank_gap"] <= MAX_RANK_GAP_FOR_CONSENSUS
]
high_pool = [
    row for row in sampling_frame
    if row["combined_rank"] >= 0.75 and row["rank_gap"] <= MAX_RANK_GAP_FOR_CONSENSUS
]
conflict_pool = [
    row for row in sampling_frame
    if row["rank_gap"] >= MIN_RANK_GAP_FOR_CONFLICT or row["score_gap"] >= MIN_SCORE_GAP_FOR_CONFLICT
]

print(f"low pool：{len(low_pool):,}")
print(f"high pool：{len(high_pool):,}")
print(f"conflict pool：{len(conflict_pool):,}")


def extremeness(row: dict, stratum: str) -> float:
    if stratum == "low":
        return 1.0 - row["combined_rank"]
    if stratum == "high":
        return row["combined_rank"]
    return row["rank_gap"] + row["score_gap"]


def greedy_diverse_select(
    pool: List[dict],
    stratum: str,
    n: int,
    used_ids: set,
    global_covered: Dict[str, set],
) -> List[dict]:
    selected = []
    local_covered = defaultdict(set)
    remaining = [row for row in pool if row["set_id"] not in used_ids]
    dims = ["temp_bin", "occasion_bin", "met_bin", "style_bin"]

    while len(selected) < n and remaining:
        best = None
        best_key = None
        for row in remaining:
            new_global = sum(1 for dim in dims if row[dim] not in global_covered[dim])
            new_local = sum(1 for dim in dims if row[dim] not in local_covered[dim])
            key = (new_global * 3 + new_local, extremeness(row, stratum), -random.random())
            if best_key is None or key > best_key:
                best_key = key
                best = row
        selected.append(best)
        used_ids.add(best["set_id"])
        for dim in dims:
            global_covered[dim].add(best[dim])
            local_covered[dim].add(best[dim])
        remaining = [row for row in remaining if row["set_id"] not in used_ids]

    if len(selected) < n:
        raise ValueError(f"{stratum} 抽樣不足 {n} 筆，目前只選到 {len(selected)} 筆。")
    return selected


used_ids = set()
global_covered = defaultdict(set)

selected_low = greedy_diverse_select(low_pool, "low", N_PER_STRATUM, used_ids, global_covered)
selected_high = greedy_diverse_select(high_pool, "high", N_PER_STRATUM, used_ids, global_covered)
selected_conflict = greedy_diverse_select(conflict_pool, "conflict", N_PER_STRATUM, used_ids, global_covered)

selected_cases = []
for stratum, rows in [
    ("low", selected_low),
    ("high", selected_high),
    ("conflict", selected_conflict),
]:
    for row in rows:
        out = dict(row)
        out["audit_stratum"] = stratum
        out["sample_rule"] = {
            "low": f"bottom 25% by combined_rank and rank_gap <= {MAX_RANK_GAP_FOR_CONSENSUS}",
            "high": f"top 25% by combined_rank and rank_gap <= {MAX_RANK_GAP_FOR_CONSENSUS}",
            "conflict": f"rank_gap >= {MIN_RANK_GAP_FOR_CONFLICT} or score_gap >= {MIN_SCORE_GAP_FOR_CONFLICT}",
        }[stratum]
        selected_cases.append(out)

selected_cases = sorted(selected_cases, key=lambda row: (row["audit_stratum"], row["set_id"]))
for i, row in enumerate(selected_cases, start=1):
    row["sample_order"] = i

write_csv(SELECTED_CASES_CSV, selected_cases)
SELECTED_CASES_JSON.write_text(json.dumps(selected_cases, ensure_ascii=False, indent=2), encoding="utf-8")
print("[寫入]", SELECTED_CASES_JSON)

print("抽樣完成：", len(selected_cases))
print("分層數量：", Counter(row["audit_stratum"] for row in selected_cases))
print("溫度覆蓋：", Counter(row["temp_bin"] for row in selected_cases))
print("場合覆蓋：", Counter(row["occasion_bin"] for row in selected_cases))
print("活動量覆蓋：", Counter(row["met_bin"] for row in selected_cases))
print("風格覆蓋：", Counter(row["style_bin"] for row in selected_cases))


low pool：4,822
high pool：5,590
conflict pool：9,967
[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A36_human_audit_30_selected_cases.csv
[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A37_human_audit_30_selected_cases.json
抽樣完成： 30
分層數量： Counter({'conflict': 10, 'high': 10, 'low': 10})
溫度覆蓋： Counter({'mild (18-24 degC)': 14, 'cold (<12 degC)': 6, 'warm/hot (>=24 degC)': 5, 'cool (12-18 degC)': 5})
場合覆蓋： Counter({'other/unclear': 9, 'evening/event': 8, 'outdoor/travel': 5, 'casual/daily': 5, 'formal/work': 3})
活動量覆蓋： Counter({'low activity (<1.3 MET)': 16, 'moderate activity (1.3-1.6 MET)': 7, 'higher activity (>=1.6 MET)': 7})
風格覆蓋： Counter({'elegant/romantic': 7, 'other/unclear': 5, 'casual/street': 5, 'boho/beach': 4, 'minimal/classic': 3, 'dark/edgy': 3, 'sporty/active': 3})


In [10]:
# ------------------------------------------------------------
# 7. Build manual audit template and paginated web page
# ------------------------------------------------------------
def checklist_weight_sum(checklist: List[dict]) -> float:
    return sum(float(item.get("weight", 1)) for item in checklist)


checklists = {
    "qwen": qwen_checklist,
    "gemma": gemma_checklist,
}


# 中文翻譯僅用於人工查核頁輔助閱讀；原始 C* checklist 英文題目仍完整保留。
CHECKLIST_TRANSLATIONS = {
    ('qwen', 'c8'): '描述是否避免加入套裝拼貼圖中沒有出現的元素或細節？',
    ('qwen', 'c1'): '描述是否包含攝氏溫度與符合套裝的冷暖感受，例如涼爽、溫暖或炎熱？',
    ('qwen', 'c2'): '描述是否指出整體穿搭風格或主題，例如海灘時尚、哥德風或運動風？',
    ('qwen', 'c4'): '描述是否指出此套裝適合的場合或情境，例如約會、婚禮或夜晚外出？',
    ('qwen', 'c3'): '描述是否提到套裝中的主要配色或關鍵顏色？',
    ('qwen', 'c5'): '描述是否捕捉到整體氛圍或美感語氣，例如輕鬆、華麗或精緻？',
    ('qwen', 'c7'): '描述是否使用精簡、適合作為搜尋或請求的查詢式語言？',
    ('qwen', 'c6'): '描述是否提到定義此套裝的主要服飾或單品，例如洋裝、比基尼或背心？',
    ('qwen', 'c9'): '描述是否能將視覺元素合理連結成一致的穿搭敘事？',
    ('qwen', 'c10'): '描述是否文法正確、語句流暢，且像自然的使用者查詢？',
    ('gemma', 'c5'): '描述是否避免對套裝做出圖片中無法直接觀察到的主張？',
    ('gemma', 'c15'): '描述是否準確呈現圖片中套裝的整體印象？',
    ('gemma', 'c3'): '描述是否包含符合套裝適穿性的溫度或季節資訊？',
    ('gemma', 'c8'): '描述是否提出適合此套裝的可能場合或活動？',
    ('gemma', 'c2'): '描述是否辨識出套裝的整體風格或美感，例如休閒、正式或波希米亞風？',
    ('gemma', 'c6'): '描述是否使用使用者搜尋相似套裝時可能會使用的語言？',
    ('gemma', 'c1'): '描述是否提到套裝中的關鍵顏色？',
    ('gemma', 'c7'): '描述是否突顯套裝中明顯的材質或圖案？',
    ('gemma', 'c4'): '描述是否使用能準確反映套裝情緒或氛圍的形容詞？',
    ('gemma', 'c10'): '描述是否文法正確且容易理解？',
    ('gemma', 'c9'): '描述是否精簡，並避免不必要的細節？',
    ('gemma', 'c11'): '描述是否聚焦於套裝整體，而不是只描述個別單品？',
    ('gemma', 'c13'): '描述是否使用與時尚和服裝相關的描述詞？',
    ('gemma', 'c12'): '描述是否避免使用可套用於許多套裝的過度籠統詞彙？',
    ('gemma', 'c16'): '描述是否提到明確的攝氏溫度？',
}


def question_zh(checklist_name: str, item: dict) -> str:
    return CHECKLIST_TRANSLATIONS.get((checklist_name, str(item["id"])), "")


def translated_checklist(checklist_name: str, checklist: list) -> list:
    rows = []
    for item in checklist:
        row = dict(item)
        row["text_zh"] = question_zh(checklist_name, item)
        rows.append(row)
    return rows


translated_checklists = {
    "qwen": translated_checklist("qwen", qwen_checklist),
    "gemma": translated_checklist("gemma", gemma_checklist),
}

template_rows = []
for case in selected_cases:
    for checklist_name, checklist in checklists.items():
        for item in checklist:
            template_rows.append({
                "reviewer": "",
                "set_id": case["set_id"],
                "sample_order": case["sample_order"],
                "audit_stratum": case["audit_stratum"],
                "checklist": checklist_name,
                "item_id": item["id"],
                "question_zh": question_zh(checklist_name, item),
                "question_en": item["text"],
                "weight": item.get("weight", 1),
                "human_answer": "",
                "human_answer_yes": "",
                "note": "",
            })

write_csv(MANUAL_TEMPLATE_CSV, template_rows)



# 生成描述中文翻譯僅用於人工查核頁輔助閱讀；英文原文仍保留為實際查核對象。
GENERATED_DESCRIPTION_TRANSLATIONS = {
    '175762493': '1.7°C，適合與朋友冬季城市漫步的舒適奢華層次穿搭',
    '187045406': '-10.0°C，適合時髦城市散步的學院風保暖層次造型',
    '206997605': '22.9°C，晚間華麗風：天鵝絨洋裝與醒目珠寶造型',
    '212444422': '21.9°C，輕鬆街頭風格：適合放鬆日常的舒適慢跑褲造型',
    '213569484': '22.0°C，暗黑星光電影夜套裝：適合電影感夜晚的舒適層次穿搭',
    '218370600': '25.8°C，暗黑加勒比神祕風：蕾絲與束身元素的度假套裝',
    '225175559': '17°C，適合節慶夏日晚間的夢幻花卉洋裝與流蘇細節造型',
    '53820793': '21.1°C，輕鬆皮拉提斯風格：適合放鬆運動的舒適層次穿搭',
    '82006848': '15.1°C，鮮明假日跑步穿搭：亮色與舒適層次造型',
    '86012987': '21.7°C，搭配丹寧與珍珠的夢幻成年禮賓客造型',
    '178417522': '16°C，舒適玉米迷宮氛圍：帶有精緻細節的嬉皮休閒造型',
    '180525625': '22.3°C，夜晚黑色電影感：皮手套與戲劇感細肩帶洋裝造型',
    '182550457': '11.0°C，舒適咖啡會面風格：豹紋高跟鞋與格紋西裝外套',
    '202122212': '5°C，適合精緻午後散步的學院風旅行套裝',
    '206715568': '25.0°C，適合暖天跑步的輕鬆運動服造型',
    '211237284': '25.0°C，輕鬆露肩風格，搭配時髦斜背包，適合放鬆外出日',
    '222746237': '22.7°C，微風海灘氛圍，搭配俏皮泳裝與草編配件',
    '224932383': '23°C，帶有貓咪元素的俏皮圓點氛圍，適合時髦微風外出日',
    '225133485': '21°C，適合精緻晚宴場合的夢幻紫色晚間造型',
    '225246911': '23°C，輕鬆有型的丹寧外套與俏皮印花，適合微風城市漫步',
    '146086172': '-38.0°C，舒適天鵝絨夜晚：戲劇感冬季套裝',
    '158633248': '6°C，夕陽漫步中的鮮明飄逸洋裝，適合溫暖夜晚冒險',
    '184137277': '55°C，穿著內搭褲與蓬蓬裙，在冰霜童話感中奔跑的造型',
    '193231221': '1.4°C，舒適冬季氛圍：深色丹寧、花卉點綴與寒冷日子的舒適 T 恤',
    '195054948': '-36.7°C，絕地武士的冬季衣櫥：適合銀河冒險的保暖層次穿搭',
    '198494269': '16°C，適合涼爽外出日的輕鬆灰色褲裝套裝時尚',
    '203011830': '5°C，適合暖天的輕鬆時髦：現代極簡造型',
    '204669155': '5°C，帶有花卉印花與陽光色調的微風夏日穿搭',
    '222196303': '5°C，條紋亞麻帶來輕鬆夏日氛圍，適合放鬆海灘日',
    '222964487': '17.2°C，適合涼爽舒適日子的輕鬆辦公室層次穿搭',
}


def generated_description_zh(case: dict) -> str:
    return GENERATED_DESCRIPTION_TRANSLATIONS.get(str(case['set_id']), '')


# 保留圖片作為人工查核依據；圖片下方只顯示 Item 編號，不顯示 item title / description。
case_payload = []
for case in selected_cases:
    image_items = []
    for idx, item in enumerate(item_summary_for_set(case["set_id"]), start=1):
        image_items.append({
            "item_id": item["item_id"],
            "index": item.get("index"),
            "image_src": item["image_src"],
            "display_label": f"Item {idx}",
        })
    case_payload.append({
        "set_id": case["set_id"],
        "sample_order": case["sample_order"],
        "audit_stratum": case["audit_stratum"],
        "temp_bin": case["temp_bin"],
        "met_bin": case["met_bin"],
        "occasion_bin": case["occasion_bin"],
        "style_bin": case["style_bin"],
        "generated_title": case["generated_title"],
        "generated_title_zh": generated_description_zh(case),
        "items": image_items,
    })




def to_js(obj) -> str:
    return json.dumps(obj, ensure_ascii=False, indent=2)


html_template = r"""<!doctype html>
<html lang="zh-Hant">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>30-case Human Audit</title>
  <style>
    :root {
      --ink: #1f2937;
      --muted: #64748b;
      --line: #d8dee8;
      --soft: #f8fafc;
      --panel: #ffffff;
      --blue: #2f77b4;
      --orange: #f28e2b;
      --green: #15803d;
      --red: #b91c1c;
    }
    * { box-sizing: border-box; }
    body {
      margin: 0;
      font-family: "Noto Sans TC", "Microsoft JhengHei", Arial, sans-serif;
      color: var(--ink);
      background: #eef2f7;
    }
    header {
      position: sticky;
      top: 0;
      z-index: 10;
      background: #ffffff;
      border-bottom: 1px solid var(--line);
      padding: 14px 20px 12px;
    }
    .header-inner {
      max-width: 1120px;
      margin: 0 auto;
      display: grid;
      grid-template-columns: minmax(0, 1fr) 280px;
      gap: 18px;
      align-items: center;
    }
    h1 { margin: 0 0 4px; font-size: 22px; }
    .subtitle { margin: 0; color: var(--muted); font-size: 13px; line-height: 1.5; }
    .progress-text { font-size: 13px; color: var(--muted); text-align: right; margin-bottom: 7px; }
    .progress-track {
      width: 100%; height: 10px; border-radius: 999px;
      background: #e2e8f0; overflow: hidden; border: 1px solid #cbd5e1;
    }
    .progress-fill { height: 100%; width: 0%; background: linear-gradient(90deg, var(--blue), var(--orange)); transition: width 0.18s ease; }
    main { max-width: 1120px; margin: 0 auto; padding: 18px 20px 28px; }
    .toolbar, .question-card { background: var(--panel); border: 1px solid var(--line); border-radius: 8px; }
    .toolbar { padding: 14px; display: grid; grid-template-columns: 220px minmax(0, 1fr) auto auto; gap: 12px; align-items: end; margin-bottom: 14px; }
    label { display: block; margin-bottom: 6px; font-size: 13px; font-weight: 800; color: #334155; }
    input { width: 100%; border: 1px solid #cbd5e1; border-radius: 6px; padding: 9px 10px; font: inherit; outline: none; }
    input:focus { border-color: var(--blue); box-shadow: 0 0 0 3px rgba(47, 119, 180, 0.12); }
    button { min-height: 38px; border: 1px solid transparent; border-radius: 6px; padding: 8px 12px; font: inherit; font-weight: 850; cursor: pointer; background: var(--blue); color: #fff; }
    button.secondary { background: #fff; color: #334155; border-color: var(--line); }
    button.orange { background: var(--orange); }
    button:disabled { opacity: 0.45; cursor: not-allowed; }
    .count-box { color: var(--muted); font-size: 13px; line-height: 1.55; align-self: center; }
    .question-card { padding: 16px; }
    .case-head { display: flex; justify-content: space-between; gap: 12px; align-items: flex-start; border-bottom: 1px solid var(--line); padding-bottom: 12px; margin-bottom: 12px; }
    .case-title { margin: 0; font-size: 18px; }
    .badges { display: flex; flex-wrap: wrap; gap: 6px; margin-top: 8px; }
    .badge { border-radius: 999px; padding: 4px 8px; background: #e2e8f0; color: #334155; font-size: 12px; font-weight: 800; white-space: nowrap; }
    .badge.conflict { background: #ffedd5; color: #9a3412; }
    .badge.high { background: #dcfce7; color: #166534; }
    .badge.low { background: #fee2e2; color: #991b1b; }
    .image-grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(96px, 1fr)); gap: 10px; margin: 12px 0 14px; }
    .image-tile { border: 1px solid var(--line); border-radius: 8px; overflow: hidden; background: #fff; }
    .image-tile img { display: block; width: 100%; height: 108px; object-fit: contain; background: #f8fafc; }
    .image-label { padding: 6px 7px; font-size: 12px; color: var(--muted); text-align: center; border-top: 1px solid #eef2f7; }
    .description { margin: 12px 0 14px; padding: 12px; border: 1px solid var(--line); border-radius: 8px; background: var(--soft); line-height: 1.65; font-size: 15px; }
    .desc-label { display: block; color: var(--muted); font-size: 12px; font-weight: 800; margin-bottom: 6px; }
    .desc-zh { font-size: 15px; font-weight: 800; color: var(--ink); margin-bottom: 4px; }
    .desc-en { font-size: 13px; color: var(--muted); }
    .lists { display: grid; grid-template-columns: 1fr 1fr; gap: 14px; }
    .list-panel { border: 1px solid var(--line); border-radius: 8px; overflow: hidden; background: #fff; }
    .list-panel h2 { margin: 0; padding: 11px 13px; background: #f8fafc; border-bottom: 1px solid var(--line); font-size: 15px; }
    .check-row { display: grid; grid-template-columns: 26px minmax(0, 1fr); gap: 8px; align-items: start; padding: 10px 12px; border-bottom: 1px solid #eef2f7; cursor: pointer; }
    .check-row:last-child { border-bottom: 0; }
    .check-row input { width: 18px; height: 18px; margin: 1px 0 0; }
    .check-text { display: grid; gap: 4px; }
    .question-zh { font-size: 13px; line-height: 1.45; font-weight: 800; color: var(--ink); }
    .question-en { font-size: 12px; line-height: 1.45; color: var(--muted); }
    .nav-row { display: flex; justify-content: space-between; align-items: center; gap: 10px; margin-top: 14px; }
    .last-count { color: var(--muted); font-size: 13px; }
    @media (max-width: 860px) {
      .header-inner { grid-template-columns: 1fr; }
      .progress-text { text-align: left; }
      .toolbar { grid-template-columns: 1fr; }
      .lists { grid-template-columns: 1fr; }
      .case-head { display: block; }
    }
  </style>
</head>
<body>
  <header>
    <div class="header-inner">
      <div>
        <h1>30-case Human Audit</h1>
        <p class="subtitle">Left and right columns show Qwen and Gemma C* checklists. Checked means YES; unchecked means NO.</p>
      </div>
      <div>
        <div class="progress-text" id="progressText">0 / 30 completed</div>
        <div class="progress-track"><div class="progress-fill" id="progressFill"></div></div>
      </div>
    </div>
  </header>

  <main>
    <div class="toolbar">
      <div>
        <label for="reviewer">Reviewer name</label>
        <input id="reviewer" value="reviewer1">
      </div>
      <div class="count-box" id="currentCount">Current case: checked 0, unchecked 0</div>
      <button class="secondary" type="button" onclick="clearSaved()">Clear saved answers</button>
      <button class="orange" type="button" onclick="exportCsv()">Export A38 CSV</button>
    </div>

    <section class="question-card">
      <div class="case-head">
        <div>
          <h2 class="case-title" id="caseTitle"></h2>
          <div class="badges" id="caseBadges"></div>
        </div>
        <div class="count-box" id="pageIndex"></div>
      </div>
      <div class="image-grid" id="outfitImages"></div>
      <div class="description">
        <span class="desc-label">Generated description / 生成描述</span>
        <div class="desc-zh" id="descriptionTextZh"></div>
        <div class="desc-en" id="descriptionText"></div>
      </div>
      <div class="lists">
        <div class="list-panel"><h2>Qwen C* checklist</h2><div id="qwenList"></div></div>
        <div class="list-panel"><h2>Gemma C* checklist</h2><div id="gemmaList"></div></div>
      </div>
      <div class="nav-row">
        <button class="secondary" type="button" id="prevBtn" onclick="goPrev()">Previous</button>
        <div class="last-count" id="lastCount">Press Next to record checked / unchecked counts for this case.</div>
        <button type="button" id="nextBtn" onclick="goNext()">Next</button>
      </div>
    </section>
  </main>

  <script>
    const cases = __CASES_JSON__;
    const checklists = __CHECKLISTS_JSON__;
    const storageKey = "human_audit_30_paginated_v1";
    let currentIndex = 0;
    let state = loadState();

    function loadState() {
      try { return JSON.parse(localStorage.getItem(storageKey)) || { answers: {}, visited: {} }; }
      catch (err) { return { answers: {}, visited: {} }; }
    }
    function saveState() { localStorage.setItem(storageKey, JSON.stringify(state)); }
    function answerKey(setId, checklist, itemId) { return `${setId}__${checklist}__${itemId}`; }
    function checkedValue(setId, checklist, itemId) { return Boolean(state.answers[answerKey(setId, checklist, itemId)]); }
    function escapeHtml(value) {
      return String(value ?? "").replaceAll("&", "&amp;").replaceAll("<", "&lt;").replaceAll(">", "&gt;").replaceAll('"', "&quot;").replaceAll("'", "&#039;");
    }
    function checklistItems(name) { return checklists[name] || []; }
    function renderImages(c) {
      const el = document.getElementById("outfitImages");
      const items = c.items || [];
      el.innerHTML = items.map(item => `
        <div class="image-tile">
          <img src="${escapeHtml(item.image_src)}" alt="${escapeHtml(item.display_label || 'Outfit item')}">
          <div class="image-label">${escapeHtml(item.display_label || 'Item')}</div>
        </div>`).join("");
    }
    function renderList(containerId, checklistName, setId) {
      const el = document.getElementById(containerId);
      el.innerHTML = checklistItems(checklistName).map(item => {
        const key = answerKey(setId, checklistName, item.id);
        const checked = checkedValue(setId, checklistName, item.id) ? "checked" : "";
        return `<label class="check-row"><input type="checkbox" data-key="${escapeHtml(key)}" ${checked} onchange="onCheckChange(event)"><span class="check-text"><span class="question-zh">${escapeHtml(item.text_zh || '')}</span><span class="question-en">${escapeHtml(item.text)}</span></span></label>`;
      }).join("");
    }
    function renderBadges(c) {
      const badgeClass = c.audit_stratum || "";
      return [
        `<span class="badge ${escapeHtml(badgeClass)}">${escapeHtml(c.audit_stratum)}</span>`,
        `<span class="badge">${escapeHtml(c.temp_bin)}</span>`,
        `<span class="badge">${escapeHtml(c.met_bin)}</span>`,
        `<span class="badge">${escapeHtml(c.occasion_bin)}</span>`,
        `<span class="badge">${escapeHtml(c.style_bin)}</span>`
      ].join("");
    }
    function countCase(c) {
      let checked = 0;
      let total = 0;
      for (const checklistName of ["qwen", "gemma"]) {
        for (const item of checklistItems(checklistName)) {
          total += 1;
          if (checkedValue(c.set_id, checklistName, item.id)) checked += 1;
        }
      }
      return { checked, unchecked: total - checked, total };
    }
    function updateProgress() {
      const visitedCount = Object.keys(state.visited || {}).filter(setId => state.visited[setId]).length;
      const pct = cases.length ? (visitedCount / cases.length) * 100 : 0;
      document.getElementById("progressText").textContent = `${visitedCount} / ${cases.length} completed`;
      document.getElementById("progressFill").style.width = `${pct}%`;
    }
    function renderCase() {
      const c = cases[currentIndex];
      document.getElementById("caseTitle").textContent = `Case ${c.sample_order} - set_id=${c.set_id}`;
      document.getElementById("pageIndex").textContent = `Page ${currentIndex + 1} / ${cases.length}`;
      document.getElementById("caseBadges").innerHTML = renderBadges(c);
      renderImages(c);
      document.getElementById("descriptionTextZh").textContent = c.generated_title_zh || "";
      document.getElementById("descriptionText").textContent = c.generated_title || "";
      renderList("qwenList", "qwen", c.set_id);
      renderList("gemmaList", "gemma", c.set_id);
      document.getElementById("prevBtn").disabled = currentIndex === 0;
      document.getElementById("nextBtn").textContent = currentIndex === cases.length - 1 ? "Finish" : "Next";
      updateCurrentCount();
      updateProgress();
    }
    function onCheckChange(event) {
      state.answers[event.target.dataset.key] = event.target.checked;
      saveState();
      updateCurrentCount();
    }
    function updateCurrentCount() {
      const counts = countCase(cases[currentIndex]);
      document.getElementById("currentCount").textContent = `Current case: checked ${counts.checked}, unchecked ${counts.unchecked}`;
    }
    function markCurrentVisited() {
      const c = cases[currentIndex];
      state.visited[c.set_id] = true;
      saveState();
      const counts = countCase(c);
      document.getElementById("lastCount").textContent = `Recorded: checked ${counts.checked}, unchecked ${counts.unchecked}`;
      updateProgress();
      return counts;
    }
    function goNext() {
      markCurrentVisited();
      if (currentIndex < cases.length - 1) { currentIndex += 1; renderCase(); }
      else { alert("All cases have been recorded. You can export A38 CSV now, or go back to revise answers."); }
    }
    function goPrev() {
      markCurrentVisited();
      if (currentIndex > 0) { currentIndex -= 1; renderCase(); }
    }
    function clearSaved() {
      if (!confirm("Clear all saved checkbox states?")) return;
      state = { answers: {}, visited: {} };
      saveState();
      currentIndex = 0;
      document.getElementById("lastCount").textContent = "Saved answers cleared.";
      renderCase();
    }
    function csvEscape(value) {
      const s = String(value ?? "");
      if (/[",\n\r]/.test(s)) return `"${s.replaceAll('"', '""')}"`;
      return s;
    }
    function exportCsv() {
      markCurrentVisited();
      const reviewer = document.getElementById("reviewer").value || "reviewer1";
      const visitedCount = Object.keys(state.visited || {}).filter(setId => state.visited[setId]).length;
      if (visitedCount < cases.length && !confirm(`Only ${visitedCount} / ${cases.length} cases were visited. Export anyway?`)) return;
      const headers = ["reviewer", "set_id", "sample_order", "audit_stratum", "checklist", "item_id", "question_zh", "question_en", "weight", "human_answer", "human_answer_yes", "note"];
      const rows = [headers];
      for (const c of cases) {
        for (const checklistName of ["qwen", "gemma"]) {
          for (const item of checklistItems(checklistName)) {
            const isChecked = checkedValue(c.set_id, checklistName, item.id);
            rows.push([reviewer, c.set_id, c.sample_order, c.audit_stratum, checklistName, item.id, item.text_zh || "", item.text, item.weight ?? 1, isChecked ? "YES" : "NO", isChecked ? "1" : "0", ""]);
          }
        }
      }
      const csv = rows.map(row => row.map(csvEscape).join(",")).join("\n");
      const blob = new Blob(["\ufeff" + csv], { type: "text/csv;charset=utf-8" });
      const url = URL.createObjectURL(blob);
      const a = document.createElement("a");
      a.href = url;
      a.download = "A38_human_audit_30_manual_results.csv";
      document.body.appendChild(a);
      a.click();
      a.remove();
      URL.revokeObjectURL(url);
    }
    renderCase();
  </script>
</body>
</html>
"""

html_text = (
    html_template
    .replace("__CASES_JSON__", to_js(case_payload))
    .replace("__CHECKLISTS_JSON__", to_js(translated_checklists))
)

FORM_HTML.write_text(html_text, encoding="utf-8")
print("[write]", FORM_HTML)


[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/human_audit_30/A38_human_audit_30_manual_results_template.csv
[write] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/human_audit_30/human_audit_30_form.html


In [11]:
# ------------------------------------------------------------
# 8. 圖表樣式：明確 XY 軸名稱與尺度
# ------------------------------------------------------------
font_candidates = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.otf",
    "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
    "/mnt/c/Windows/Fonts/msjh.ttc",
]

selected_font = None
for font_path in font_candidates:
    if Path(font_path).exists():
        fm.fontManager.addfont(font_path)
        selected_font = fm.FontProperties(fname=font_path).get_name()
        break

if selected_font is None:
    selected_font = "DejaVu Sans"

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [selected_font, "DejaVu Sans"],
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "axes.unicode_minus": False,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
})

BLUE = "#2f77b4"
ORANGE = "#f28e2b"
GREY = "#8a8a8a"
RED = "#d65f4c"


def save_figure(fig, basename: str):
    png = FIG_DIR / f"{basename}.png"
    svg = FIG_DIR / f"{basename}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("[寫入]", png)
    print("[寫入]", svg)


def wrap_label(text: str, max_len: int = 34) -> str:
    text = str(text)
    if len(text) <= max_len:
        return text
    words = text.split()
    lines = []
    cur = ""
    for word in words:
        if len(cur) + len(word) + 1 > max_len and cur:
            lines.append(cur)
            cur = word
        else:
            cur = f"{cur} {word}".strip()
    if cur:
        lines.append(cur)
    return "\n".join(lines)


def short_coverage_label(value: str) -> str:
    mapping = {
        "conflict": "Conflict",
        "high": "High score",
        "low": "Low score",
        "cold (<12 degC)": "Cold\n(<12 degC)",
        "cool (12-18 degC)": "Cool\n(12-18 degC)",
        "mild (18-24 degC)": "Mild\n(18-24 degC)",
        "warm/hot (>=24 degC)": "Warm/hot\n(>=24 degC)",
        "low activity (<1.3 MET)": "Low\n(<1.3 MET)",
        "moderate activity (1.3-1.6 MET)": "Moderate\n(1.3-1.6 MET)",
        "higher activity (>=1.6 MET)": "Higher\n(>=1.6 MET)",
        "other/unclear": "Other/\nunclear",
        "outdoor/travel": "Outdoor/\ntravel",
        "evening/event": "Evening/\nevent",
        "casual/daily": "Casual/\ndaily",
        "formal/work": "Formal/\nwork",
        "elegant/romantic": "Elegant/\nromantic",
        "casual/street": "Casual/\nstreet",
        "boho/beach": "Boho/\nbeach",
        "minimal/classic": "Minimal/\nclassic",
        "dark/edgy": "Dark/\nedgy",
        "sporty/active": "Sporty/\nactive",
    }
    return mapping.get(str(value), str(value))


def write_sampling_coverage_outputs():
    """輸出 30 筆人工查核樣本的情境覆蓋摘要；不需要人工 YES/NO 結果。"""
    coverage_rows = []
    for dim in ["audit_stratum", "temp_bin", "met_bin", "occasion_bin", "style_bin"]:
        counts = Counter(row[dim] for row in selected_cases)
        for value, count in sorted(counts.items()):
            coverage_rows.append({
                "dimension": dim,
                "value": value,
                "n_samples": count,
                "ratio": f"{count / len(selected_cases):.4f}",
            })
    write_csv(DERIVED_DIR / "A42_human_audit_30_coverage_summary.csv", coverage_rows)

    fig, axes = plt.subplots(2, 2, figsize=(8.6, 5.8))
    coverage_dims = [
        ("temp_bin", "Temperature (degC)"),
        ("met_bin", "Activity level (MET)"),
        ("occasion_bin", "Occasion type"),
        ("style_bin", "Style type"),
    ]
    for ax, (dim, x_label) in zip(axes.ravel(), coverage_dims):
        counts = Counter(row[dim] for row in selected_cases)
        labels = list(counts.keys())
        values = [counts[label] for label in labels]
        colors = [BLUE for label in labels]
        bars = ax.bar(range(len(labels)), values, color=colors, edgecolor="black", linewidth=0.55)
        ax.set_xlabel(x_label, fontsize=10)
        ax.set_ylabel("Samples (n)", fontsize=10)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels([short_coverage_label(label) for label in labels], fontsize=8, rotation=0, ha="center")
        ax.tick_params(axis="y", labelsize=9)
        ax.set_ylim(0, max(values) + 2)
        ax.grid(axis="y", linestyle="--", linewidth=0.45, alpha=0.32)
        ax.set_axisbelow(True)
        for bar, value in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, value + 0.12, str(value), ha="center", va="bottom", fontsize=8)
    fig.tight_layout(pad=1.05)
    save_figure(fig, "F30_human_audit30_sampling_coverage")
    plt.close(fig)

write_sampling_coverage_outputs()


[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A42_human_audit_30_coverage_summary.csv


[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/figures/F30_human_audit30_sampling_coverage.png
[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/figures/F30_human_audit30_sampling_coverage.svg


In [12]:
# ------------------------------------------------------------
# 9. 若人工結果存在，讀取並分析；若尚未填寫則略過
# ------------------------------------------------------------
def answer_to_float(value) -> Optional[float]:
    s = str(value).strip().lower()
    if s in {"yes", "y", "1", "true", "是"}:
        return 1.0
    if s in {"no", "n", "0", "false", "否"}:
        return 0.0
    return None


def weighted_score(decisions: Dict[str, float], checklist: List[dict]) -> Optional[float]:
    total_w = 0.0
    total = 0.0
    for item in checklist:
        item_id = str(item["id"])
        if item_id not in decisions:
            continue
        w = float(item.get("weight", 1))
        total += w * float(decisions[item_id])
        total_w += w
    if total_w == 0:
        return None
    return total / total_w


def pearson(xs: List[float], ys: List[float]) -> Optional[float]:
    if len(xs) < 2:
        return None
    mx, my = mean(xs), mean(ys)
    vx = sum((x - mx) ** 2 for x in xs)
    vy = sum((y - my) ** 2 for y in ys)
    if vx == 0 or vy == 0:
        return None
    return sum((x - mx) * (y - my) for x, y in zip(xs, ys)) / math.sqrt(vx * vy)


def rank_values(values: List[float]) -> List[float]:
    indexed = sorted(enumerate(values), key=lambda kv: kv[1])
    ranks = [0.0] * len(values)
    i = 0
    while i < len(indexed):
        j = i
        while j + 1 < len(indexed) and indexed[j + 1][1] == indexed[i][1]:
            j += 1
        avg = (i + 1 + j + 1) / 2
        for k in range(i, j + 1):
            ranks[indexed[k][0]] = avg
        i = j + 1
    return ranks


def spearman(xs: List[float], ys: List[float]) -> Optional[float]:
    if len(xs) < 2:
        return None
    return pearson(rank_values(xs), rank_values(ys))


def sd(values: List[float]) -> Optional[float]:
    return stdev(values) if len(values) >= 2 else 0.0 if values else None


def fmt(value, digits: int = 4):
    if value is None:
        return "NA"
    return f"{value:.{digits}f}"


def model_item_decisions(set_id: str, checklist_name: str) -> Dict[str, float]:
    source = qwen_scores if checklist_name == "qwen" else gemma_scores
    return {
        item_id: 1.0 if ans == "yes" else 0.0
        for item_id, ans in source[set_id]["decisions"].items()
        if ans in {"yes", "no"}
    }


def model_score(set_id: str, checklist_name: str) -> float:
    source = qwen_scores if checklist_name == "qwen" else gemma_scores
    return float(source[set_id]["score"])


if not MANUAL_RESULTS_CSV.exists():
    print("\n尚未找到人工結果 CSV，因此先停止於抽樣與網頁產生階段。")
    print("請打開：", FORM_HTML)
    print("填寫後匯出 CSV，並放到：", MANUAL_RESULTS_CSV)
else:
    manual_rows = read_csv_rows(MANUAL_RESULTS_CSV)
    selected_by_id = {row["set_id"]: row for row in selected_cases}
    selected_ids = [row["set_id"] for row in selected_cases]

    # reviewer 可能多位；同一 set_id/checklist/item 先取 reviewer 平均。
    human_item_values = defaultdict(list)
    for row in manual_rows:
        set_id = str(row.get("set_id", ""))
        if set_id not in selected_by_id:
            continue
        checklist_name = str(row.get("checklist", "")).strip().lower()
        item_id = str(row.get("item_id", "")).strip()
        ans = answer_to_float(row.get("human_answer_yes", row.get("human_answer", "")))
        if ans is None:
            ans = answer_to_float(row.get("human_answer", ""))
        if checklist_name in {"qwen", "gemma"} and item_id and ans is not None:
            human_item_values[(set_id, checklist_name, item_id)].append(ans)

    human_item_mean = {
        key: mean(values)
        for key, values in human_item_values.items()
        if values
    }

    missing_count = 0
    human_score_by_case = {}
    for set_id in selected_ids:
        for checklist_name, checklist in checklists.items():
            decisions = {}
            for item in checklist:
                key = (set_id, checklist_name, str(item["id"]))
                if key in human_item_mean:
                    decisions[str(item["id"])] = human_item_mean[key]
                else:
                    missing_count += 1
            human_score_by_case[(set_id, checklist_name)] = weighted_score(decisions, checklist)

    print(f"人工結果列數：{len(manual_rows):,}")
    print(f"缺漏的 set/checklist/item 判斷數：{missing_count:,}")

    # --------------------------------------------------------
    # 9-1. 模型與人工加權分數比較
    # --------------------------------------------------------
    metric_rows = []
    for checklist_name in ["qwen", "gemma"]:
        xs_human = []
        ys_model = []
        for set_id in selected_ids:
            h = human_score_by_case.get((set_id, checklist_name))
            if h is None:
                continue
            xs_human.append(float(h))
            ys_model.append(model_score(set_id, checklist_name))

        diffs = [m - h for h, m in zip(xs_human, ys_model)]
        abs_diffs = [abs(d) for d in diffs]
        sq_diffs = [d * d for d in diffs]
        metric_rows.append({
            "scope": "30_case_human_audit",
            "checklist": checklist_name,
            "n": len(xs_human),
            "bias_model_minus_human": fmt(mean(diffs) if diffs else None),
            "mae": fmt(mean(abs_diffs) if abs_diffs else None),
            "rmse": fmt(math.sqrt(mean(sq_diffs)) if sq_diffs else None),
            "pearson": fmt(pearson(xs_human, ys_model)),
            "spearman": fmt(spearman(xs_human, ys_model)),
            "human_mean": fmt(mean(xs_human) if xs_human else None),
            "model_mean": fmt(mean(ys_model) if ys_model else None),
            "human_sd": fmt(sd(xs_human)),
            "model_sd": fmt(sd(ys_model)),
        })

    write_csv(TABLE_DIR / "T30_human_audit30_model_human_score_metrics.csv", metric_rows)
    write_csv(DERIVED_DIR / "A39_human_audit_30_model_human_score_metrics.csv", metric_rows)

    # --------------------------------------------------------
    # 9-2. item-level disagreement
    # --------------------------------------------------------
    item_rows = []
    for checklist_name, checklist in checklists.items():
        total_weight = checklist_weight_sum(checklist)
        for item in checklist:
            item_id = str(item["id"])
            w = float(item.get("weight", 1))
            values = []
            signed_values = []
            for set_id in selected_ids:
                h = human_item_mean.get((set_id, checklist_name, item_id))
                m = model_item_decisions(set_id, checklist_name).get(item_id)
                if h is None or m is None:
                    continue
                values.append(abs(m - h))
                signed_values.append(m - h)
            item_disagreement = mean(values) if values else None
            item_rows.append({
                "checklist": checklist_name,
                "item_id": item_id,
                "category": item.get("category", ""),
                "weight": w,
                "n": len(values),
                "item_disagreement_rate_0_1": fmt(item_disagreement),
                "weighted_item_disagreement_0_1": fmt(item_disagreement * (w / total_weight) if item_disagreement is not None else None),
                "mean_signed_model_minus_human": fmt(mean(signed_values) if signed_values else None),
                "question_zh": question_zh(checklist_name, item) if "question_zh" in globals() else "",
                "question_en": item.get("text", ""),
                "short_label": f"{checklist_name.capitalize()} {item_id}",
            })

    item_rows_sorted = sorted(
        item_rows,
        key=lambda row: safe_float(row["weighted_item_disagreement_0_1"], -1),
        reverse=True,
    )
    for idx, row in enumerate(item_rows_sorted, start=1):
        row["rank"] = idx

    write_csv(TABLE_DIR / "T31_human_audit30_item_disagreement.csv", item_rows_sorted)
    write_csv(DERIVED_DIR / "A40_human_audit_30_item_disagreement.csv", item_rows_sorted)

    # --------------------------------------------------------
    # 9-3. sample-level summary
    # --------------------------------------------------------
    sample_rows = []
    for case in selected_cases:
        set_id = case["set_id"]
        out = {
            "set_id": set_id,
            "sample_order": case["sample_order"],
            "audit_stratum": case["audit_stratum"],
            "temperature_c": case["temperature_c"],
            "temp_bin": case["temp_bin"],
            "met_value": case["met_value"],
            "met_bin": case["met_bin"],
            "occasion_bin": case["occasion_bin"],
            "style_bin": case["style_bin"],
            "generated_title": case["generated_title"],
        }
        for checklist_name in ["qwen", "gemma"]:
            h = human_score_by_case.get((set_id, checklist_name))
            m = model_score(set_id, checklist_name)
            out[f"human_{checklist_name}_score"] = fmt(h)
            out[f"model_{checklist_name}_score"] = fmt(m)
            out[f"error_{checklist_name}_model_minus_human"] = fmt(m - h if h is not None else None)
        sample_rows.append(out)

    write_csv(DERIVED_DIR / "A41_human_audit_30_sample_score_summary.csv", sample_rows)

    # --------------------------------------------------------
    # 9-4. coverage summary
    # --------------------------------------------------------
    coverage_rows = []
    for dim in ["audit_stratum", "temp_bin", "met_bin", "occasion_bin", "style_bin"]:
        counts = Counter(row[dim] for row in selected_cases)
        for value, count in sorted(counts.items()):
            coverage_rows.append({
                "dimension": dim,
                "value": value,
                "n_samples": count,
                "ratio": fmt(count / len(selected_cases)),
            })
    write_csv(DERIVED_DIR / "A42_human_audit_30_coverage_summary.csv", coverage_rows)

    # --------------------------------------------------------
    # 9-5. 圖表：抽樣覆蓋、誤差、散點、item disagreement
    # --------------------------------------------------------
    write_sampling_coverage_outputs()

    metric_map = {row["checklist"]: row for row in metric_rows}
    metric_labels = ["Bias", "MAE", "RMSE"]
    metric_keys = ["bias_model_minus_human", "mae", "rmse"]
    all_metric_values = []
    for checklist_name in ["qwen", "gemma"]:
        all_metric_values.extend([safe_float(metric_map[checklist_name][key], 0.0) for key in metric_keys])
    y_min = min(0.0, min(all_metric_values)) - 0.045
    y_max = max(0.0, max(all_metric_values)) + 0.045

    x = np.arange(len(metric_labels))
    width = 0.32
    fig, ax = plt.subplots(figsize=(5.8, 3.4))
    for idx, (checklist_name, color, label) in enumerate([("qwen", BLUE, "Qwen"), ("gemma", ORANGE, "Gemma")]):
        values = [safe_float(metric_map[checklist_name][key], 0.0) for key in metric_keys]
        offset = (idx - 0.5) * width
        bars = ax.bar(x + offset, values, width=width, label=label, color=color, edgecolor="black", linewidth=0.55)
        for bar, value in zip(bars, values):
            va = "bottom" if value >= 0 else "top"
            delta = 0.010 if value >= 0 else -0.010
            ax.text(bar.get_x() + bar.get_width() / 2, value + delta, f"{value:.3f}", ha="center", va=va, fontsize=8)
    ax.axhline(0, color="black", linewidth=0.75)
    ax.set_xlabel("Error metric", fontsize=10)
    ax.set_ylabel("Score difference / error (0-1)", fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels, fontsize=9)
    ax.tick_params(axis="y", labelsize=9)
    ax.set_ylim(y_min, y_max)
    ax.grid(axis="y", linestyle="--", linewidth=0.45, alpha=0.32)
    ax.legend(frameon=True, fontsize=8, loc="upper left", ncol=2)
    fig.tight_layout(pad=0.9)
    save_figure(fig, "F31_human_audit30_model_human_error_metrics")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(4.7, 4.2))
    for checklist_name, color, label, marker in [("qwen", BLUE, "Qwen", "o"), ("gemma", ORANGE, "Gemma", "s")]:
        xs = []
        ys = []
        for set_id in selected_ids:
            h = human_score_by_case.get((set_id, checklist_name))
            if h is None:
                continue
            xs.append(h)
            ys.append(model_score(set_id, checklist_name))
        ax.scatter(xs, ys, label=label, s=30, marker=marker, alpha=0.68, color=color, edgecolor="white", linewidth=0.45)
    ax.plot([0, 1], [0, 1], color=GREY, linestyle="--", linewidth=1.0)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_xlabel("Human weighted score (0-1)", fontsize=10)
    ax.set_ylabel("Model score (0-1)", fontsize=10)
    ax.tick_params(labelsize=9)
    ax.grid(True, linestyle="--", linewidth=0.45, alpha=0.28)
    ax.legend(frameon=True, fontsize=8, loc="lower right")
    fig.tight_layout(pad=0.9)
    save_figure(fig, "F32_human_audit30_model_human_score_scatter")
    plt.close(fig)

    top_items = list(reversed(item_rows_sorted[:12]))
    labels = [row.get("short_label") or f"{row['checklist'].capitalize()} {row['item_id']}" for row in top_items]
    values = [safe_float(row["weighted_item_disagreement_0_1"], 0.0) for row in top_items]
    colors = [ORANGE if str(row["checklist"]).lower() == "gemma" else BLUE for row in top_items]
    fig, ax = plt.subplots(figsize=(5.8, 4.4))
    bars = ax.barh(range(len(labels)), values, color=colors, edgecolor="black", linewidth=0.55)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("Weighted item disagreement rate (0-1)", fontsize=10)
    ax.set_ylabel("Checklist item ID", fontsize=10)
    ax.tick_params(axis="x", labelsize=9)
    xmax = max(values) * 1.22 if values else 0.1
    ax.set_xlim(0, xmax)
    ax.grid(axis="x", linestyle="--", linewidth=0.45, alpha=0.32)
    ax.set_axisbelow(True)
    for bar, value in zip(bars, values):
        ax.text(value + xmax * 0.015, bar.get_y() + bar.get_height() / 2, f"{value:.3f}", va="center", ha="left", fontsize=8)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor=BLUE, edgecolor="black", label="Qwen"), Patch(facecolor=ORANGE, edgecolor="black", label="Gemma")], frameon=True, fontsize=8, loc="lower right")
    fig.tight_layout(pad=0.9)
    save_figure(fig, "F33_human_audit30_item_disagreement_ranking")
    plt.close(fig)

    interpretation = [
        "A43 30 筆人工查核結果說明",
        "",
        f"本次人工查核共納入 {len(selected_cases)} 筆樣本，分為 low、high、conflict 三組，各 {N_PER_STRATUM} 筆。",
        "抽樣時同時標記並檢查溫度區間、活動量 MET 區間、場合類型與風格類型，以回應人工樣本需涵蓋不同情境條件的建議。",
        "人工查核方式沿用既有 Qwen / Gemma C* checklist，逐項判斷 YES / NO；本研究未新增另一份內容重複的查核清單。",
        "",
        "圖表座標軸修正：",
        "F30 的 x 軸為抽樣/情境類別，y 軸為人工查核樣本數 n。",
        "F31 的 x 軸為誤差指標，y 軸為模型分數與人工加權分數之差異或誤差，尺度為 0–1 score scale。",
        "F32 的 x 軸為人工加權分數，y 軸為模型分數，兩者皆為 0–1。",
        "F33 的 x 軸為加權 item-level disagreement rate，尺度為 0–1；y 軸為 C* checklist item。",
    ]
    interp_path = DERIVED_DIR / "A43_human_audit_30_interpretation.txt"
    interp_path.write_text("\n".join(interpretation), encoding="utf-8")
    print("[寫入]", interp_path)


人工結果列數：750
缺漏的 set/checklist/item 判斷數：0
[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/tables/T30_human_audit30_model_human_score_metrics.csv
[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A39_human_audit_30_model_human_score_metrics.csv
[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/tables/T31_human_audit30_item_disagreement.csv
[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A40_human_audit_30_item_disagreement.csv
[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A41_human_audit_30_sample_score_summary.csv
[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A42_human_audit_30_coverage_summary.csv
[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A42_human_audit_30_coverage_

[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/figures/F30_human_audit30_sampling_coverage.png
[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/figures/F30_human_audit30_sampling_coverage.svg


[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/figures/F31_human_audit30_model_human_error_metrics.png
[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/figures/F31_human_audit30_model_human_error_metrics.svg


[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/figures/F32_human_audit30_model_human_score_scatter.png
[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/figures/F32_human_audit30_model_human_score_scatter.svg


[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/figures/F33_human_audit30_item_disagreement_ranking.png
[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/figures/F33_human_audit30_item_disagreement_ranking.svg
[寫入] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A43_human_audit_30_interpretation.txt
